[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/causality-discovery/supervised_baseline/supervised_baseline.ipynb)

## Preliminary step

In [1]:
"""
This is a basic example of what you need to do to enter the crunch.
The code will not have access to the internet (or any socket related operation).
"""

import functools
import os
import typing
import re

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import pickle
from tqdm.auto import tqdm

from scipy.stats import spearmanr, pearsonr, ttest_rel
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score  # Replace with appropriate metric
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors

from scipy.stats import energy_distance
from scipy.stats import entropy
from scipy.stats import ttest_rel

from statsmodels.sandbox.stats.runs import runstest_1samp

import pingouin as pg  
import dcor
import lightgbm as lgb

/home/endrem/prosjekt/causalDiscovery/causalEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def graph_nodes_representation(graph, nodelist):
    """
    Create an alternative representation of a graph which is hashable
    and equivalent graphs have the same hash.

    Python cannot PROPERLY use nx.Graph/DiGraph as key for
    dictionaries, because two equivalent graphs with just different
    order of the nodes would result in different keys. This is
    undesirable here.

    So here we transform the graph into an equivalent form that is
    based on a specific nodelist and that is hashable. In this way,
    two equivalent graphs, once transformed, will result in identical
    keys.

    So we use the following trick: extract the adjacency matrix
    (with nodes in a fixed order) and then make a hashable thing out
    of it, through tuple(array.flatten()):
    """

    # This get the adjacency matrix with nodes in a given order, as
    # numpy array (which is not hashable):
    adjacency_matrix = nx.adjacency_matrix(graph, nodelist=nodelist).todense()

    # This transforms the numpy array into a hashable object:
    hashable = tuple(adjacency_matrix.flatten())

    return hashable

In [4]:
def create_graph_label():
    """
    Create a dictionary from graphs to labels, in two formats.
    """
    graph_label = {
        nx.DiGraph([("X", "Y"), ("v", "X"), ("v", "Y")]): "Confounder",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("Y", "v")]): "Collider",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("v", "Y")]): "Mediator",
        nx.DiGraph([("X", "Y"), ("v", "X")]):             "Cause of X",
        nx.DiGraph([("X", "Y"), ("v", "Y")]):             "Cause of Y",
        nx.DiGraph([("X", "Y"), ("X", "v")]):             "Consequence of X",
        nx.DiGraph([("X", "Y"), ("Y", "v")]):             "Consequence of Y",
        nx.DiGraph({"X": ["Y"], "v": []}):                "Independent",
    }

    nodelist = ["v", "X", "Y"]

    # This is an equivalent alternative to graph_label but in a form
    # for which two equivalent graphs have the same key:
    adjacency_label = {
        graph_nodes_representation(graph, nodelist): label
        for graph, label in graph_label.items()
    }

    return graph_label, adjacency_label

In [5]:
def get_labels(adjacency_matrix, adjacency_label):
    """
    Transform an adjacency_matrix (as pd.DataFrame) into a dictionary of variable:label
    """

    result = {}
    for variable in adjacency_matrix.columns.drop(["X", "Y"]):
        submatrix = adjacency_matrix.loc[[variable, "X", "Y"], [variable, "X", "Y"]]  # this is not hashable
        key = tuple(submatrix.values.flatten())  # this is hashable and a compatible with adjacency_label

        result[variable] = adjacency_label[key]

    return result

Example 1: manually creating the adjacency matrix.

In [6]:
graph_label, adjacency_label = create_graph_label()

Example 2: loading an adjacency matrix from the training set.

In [2]:
X_train = pd.read_pickle('./X_train.pkl')
y_train = pd.read_pickle('./y_train.pkl')
X_test = pd.read_pickle('./X_test.pkl')

with open('X_y_group_train.pkl', 'rb') as f:
    X_y_group_train_loaded = pickle.load(f)

### Correlation based features
- pearson_correlation
- spearman_correlation
- kendall_correlation
- pearson_correlation_with_rolling_multi_labeled
- partial_correlation
- partial_correlation_rank

**Distance Correlation Coefficient:** A nice feature with about **1 point** imporvement.
    - ***Func* dcor**: Provides the distance correlation coefficient.
    Reference: https://github.com/vnmabus/dcor

In [8]:
def pearson_correlation(dataset, the_method='pearson'):
    """
    Given a dataset, we compute the correlation-based features for each
    varibale, which are the correlation between that variable with X and Y,
    as well as summary statistics (max, min, mean, std) of all pairs
    of correlations.
    """

    corr_matrix = dataset.corr(method=the_method)

    variables = dataset.columns.drop(["X", "Y"])
    df = []

    for variable in variables:
        other_corrs = corr_matrix.loc[variable].drop([variable, 'X', 'Y']).dropna() #, errors='ignore')

        corr_v_X = corr_matrix.loc[variable, 'X']
        corr_v_Y = corr_matrix.loc[variable, 'Y']

        # Handle possible NaN values (e.g., when variance is zero)
        corr_v_X = 0 if np.isnan(corr_v_X) else corr_v_X
        corr_v_Y = 0 if np.isnan(corr_v_Y) else corr_v_Y
        max_corr = 0 if other_corrs.empty else other_corrs.max()
        min_corr = 0 if other_corrs.empty else other_corrs.min()
        mean_corr = 0 if other_corrs.empty else other_corrs.mean()
        std_corr = 0 if other_corrs.empty else other_corrs.std()
        std25_corr = 0 if other_corrs.empty else np.percentile(other_corrs.values, 25)
        std75_corr = 0 if other_corrs.empty else np.percentile(other_corrs.values, 75)
        median_corr = 0 if other_corrs.empty else other_corrs.median()

        df.append({
            "variable": variable,
            f"corr(v,X)_{the_method}": corr_v_X,
            f"corr(v,Y)_{the_method}": corr_v_Y,
            f"max(corr(v, others))_{the_method}": max_corr,
            f"min(corr(v, others))_{the_method}": min_corr,
            f"mean(corr(v, others))_{the_method}": mean_corr,
            f"std(corr(v, others))_{the_method}": std_corr,
            f"std25(corr(v, others))_{the_method}": std25_corr,
            f"std75(corr(v, others))_{the_method}": std75_corr,
            f"median(corr(v, others))_{the_method}": median_corr
            #f"corr(X,Y)_{the_method}": corr_x_y
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    corr_X_Y = corr_matrix.loc['X', 'Y']
    df[f"corr(X,Y)_{the_method}"] = 0 if np.isnan(corr_X_Y) else corr_X_Y

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

In [9]:
def spearman_correlation(dataset):
    return pearson_correlation(dataset, 'spearman')

In [10]:
def kendall_correlation(dataset):
    return pearson_correlation(dataset, 'kendall')

In [3]:
def pearson_correlation_with_rolling_multi_labeled(dataset, window_sizes=[300, 600, 900], meth='pearson'):
    """
    Compute rolling correlation statistics for each variable with respect to X and Y
    over rolling windows of specified sizes.
    Each statistic is labeled with _<window_size>.
    """
    variables = dataset.columns.drop(["X", "Y"])
    
    # Initialize a dictionary to hold results for each variable
    results = {variable: {"variable": variable} for variable in variables}

    for window_size in window_sizes:
        # Get the data for the current window size
        window_data = dataset.iloc[-window_size:]

        # Compute the correlation matrix once per window size
        corr_matrix = window_data.corr(method=meth)

        corr_X_Y = corr_matrix.loc['X', 'Y']
        corr_X_Y = 0 if np.isnan(corr_X_Y) else corr_X_Y

        for variable in variables:
            # Get correlations of 'variable' with others, excluding itself, 'X', and 'Y'
            if variable in corr_matrix.index:
                other_corrs = corr_matrix.loc[variable].drop([variable, 'X', 'Y'], errors='ignore')
            else:
                other_corrs = pd.Series(dtype=float)

            # Extract required correlations with 'X' and 'Y'
            corr_v_X = corr_matrix.loc[variable, 'X'] if 'X' in corr_matrix.columns else np.nan
            corr_v_Y = corr_matrix.loc[variable, 'Y'] if 'Y' in corr_matrix.columns else np.nan

            # Handle possible NaN values (e.g., when variance is zero)
            corr_v_X = 0 if np.isnan(corr_v_X) else corr_v_X
            corr_v_Y = 0 if np.isnan(corr_v_Y) else corr_v_Y

            # Drop NaN values from other_corrs
            valid_other_corrs = other_corrs.dropna()

            # Calculate statistics for other correlations
            if valid_other_corrs.empty:
                max_corr = min_corr = mean_corr = std_corr = 0
                std25_corr = std75_corr = median_corr = 0
            else:
                max_corr = valid_other_corrs.max()
                min_corr = valid_other_corrs.min()
                mean_corr = valid_other_corrs.mean()
                std_corr = valid_other_corrs.std()
                std25_corr = np.percentile(valid_other_corrs.values, 25)
                std75_corr = np.percentile(valid_other_corrs.values, 75)
                median_corr = valid_other_corrs.median()

            # Update the results dictionary for this variable
            results[variable][f"corr(v,X)_{window_size}"] = corr_v_X
            results[variable][f"corr(v,Y)_{window_size}"] = corr_v_Y
            results[variable][f"corr(X,Y)_{window_size}"] = corr_X_Y
            results[variable][f"max(corr(v, others))_{window_size}"] = max_corr
            results[variable][f"min(corr(v, others))_{window_size}"] = min_corr
            results[variable][f"mean(corr(v, others))_{window_size}"] = mean_corr
            results[variable][f"std(corr(v, others))_{window_size}"] = std_corr
            results[variable][f"25th(corr(v, others))_{window_size}"] = std25_corr
            results[variable][f"75th(corr(v, others))_{window_size}"] = std75_corr
            results[variable][f"median(corr(v, others))_{window_size}"] = median_corr

    # Convert the results dictionary to a DataFrame
    result_df = pd.DataFrame.from_dict(results, orient='index').reset_index(drop=True)
    return result_df

In [10]:
def partial_correlation(dataset):
    """
    Computes the partial correlation coefficient between two variables given a third variable.
    Features:
    - partial_corr(v, X | Y)
    - partial_corr(v, Y | X)
    - partial_corr(X, Y | v)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        # Initialize partial correlations
        pcorr_v_X_given_Y = np.nan
        pcorr_v_Y_given_X = np.nan
        pcorr_X_Y_given_v = np.nan

        # Check for zero variance in variables
        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            print(f"Zero variance detected in variable: {variable}")
            # Assign correlations as NaN or zero
            pcorr_v_X_given_Y = 0
            pcorr_v_Y_given_X = 0
            pcorr_X_Y_given_v = 0
        else:
            # Calculate partial correlation matrix
            pcorr_matrix = dataset[[variable, 'X', 'Y']].pcorr()

            # Extract partial correlations
            pcorr_v_X_given_Y = pcorr_matrix.loc[variable, 'X']
            pcorr_v_Y_given_X = pcorr_matrix.loc[variable, 'Y']
            pcorr_X_Y_given_v = pcorr_matrix.loc['X', 'Y']

        df_list.append({
            "variable": variable,
            "partial_corr(v,X|Y)": pcorr_v_X_given_Y,
            "partial_corr(v,Y|X)": pcorr_v_Y_given_X,
            "partial_corr(X,Y|v)": pcorr_X_Y_given_v
        })

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)

    # Reorder columns
    df = df[["dataset", "variable", "partial_corr(v,X|Y)", "partial_corr(v,Y|X)", "partial_corr(X,Y|v)"]]

    # Fill NaN values with 0
    df.fillna(0, inplace=True)

    return df

In [13]:
def partial_correlation_rank(dataset):
    """
    Computes partial correlation features using Spearman's rank correlation.
    Features:
    - partial_corr_rank(v, X | Y)
    - partial_corr_rank(v, Y | X)
    - partial_corr_rank(X, Y | v)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    for variable in variables:
        pcorr_v_X_given_Y = 0
        pcorr_v_Y_given_X = 0
        pcorr_X_Y_given_v = 0

        # Check for zero variance in variables
        if (dataset[variable].std() == 0 or
            dataset["X"].std() == 0 or
            dataset["Y"].std() == 0):
            pass  # Correlations remain zero
        else:
            # Compute partial correlations using pingouin with Spearman method
            pcorr_v_X_given_Y = pg.partial_corr(data=dataset, x=variable, y='X', covar='Y', method='spearman')['r'].values[0]
            pcorr_v_Y_given_X = pg.partial_corr(data=dataset, x=variable, y='Y', covar='X', method='spearman')['r'].values[0]
            pcorr_X_Y_given_v = pg.partial_corr(data=dataset, x='X', y='Y', covar=variable, method='spearman')['r'].values[0]

        # Append results
        df_list.append({
            "variable": variable,
            "partial_corr_rank(v,X|Y)": pcorr_v_X_given_Y,
            "partial_corr_rank(v,Y|X)": pcorr_v_Y_given_X,
            "partial_corr_rank(X,Y|v)": pcorr_X_Y_given_v
        })

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", None)
    df = df[["dataset", "variable", "partial_corr_rank(v,X|Y)", "partial_corr_rank(v,Y|X)", "partial_corr_rank(X,Y|v)"]]
    df.fillna(0, inplace=True)

    return df

In [14]:
def distance_correlation_features(dataset):
    """
    Computes distance-correlation-based features for each variable, including distance
    correlation with X and Y, summary statistics of distance correlation with other variables,
    and distance correlation between X and Y.
    
    Features:
    - dcor(v, X)
    - dcor(v, Y)
    - dcor(v, X)^2
    - dcor(v, Y)^2
    - max(dcor(v, others))
    - min(dcor(v, others))
    - mean(dcor(v, others))
    - std(dcor(v, others))
    - 25%(dcor(v, others))
    - 75%(dcor(v, others))
    - dcor(X, Y)
    - dcor(X, Y)^2
    """
    # Define variables excluding 'X' and 'Y'
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    # Handle missing values by imputing with the mean
    imputer = SimpleImputer(strategy='mean')
    imputed_data = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

    # Scale the data
    scaler = StandardScaler()
    scaled_data = pd.DataFrame(scaler.fit_transform(imputed_data), columns=dataset.columns)

    # Compute distance correlation between X and Y
    dcor_X_Y = dcor.distance_correlation(scaled_data["X"], scaled_data["Y"])
    dcor_X_Y_sq = dcor_X_Y ** 2

    for variable in variables:
        # Compute distance correlation between v and X
        dcor_v_X = dcor.distance_correlation(scaled_data[variable], scaled_data["X"])
        dcor_v_X_sq = dcor_v_X ** 2

        # Compute distance correlation between v and Y
        dcor_v_Y = dcor.distance_correlation(scaled_data[variable], scaled_data["Y"])
        dcor_v_Y_sq = dcor_v_Y ** 2

        # Compute distance correlation between v and all other variables (excluding X and Y)
        other_vars = variables.drop(variable)
        dcor_v_others = []
        for other_var in other_vars:
            dcor_val = dcor.distance_correlation(scaled_data[variable], scaled_data[other_var])
            dcor_v_others.append(dcor_val)
        
        # Handle cases where there might be no other variables
        if dcor_v_others:
            max_dcor = max(dcor_v_others)
            min_dcor = min(dcor_v_others)
            mean_dcor = np.mean(dcor_v_others)
            std_dcor = np.std(dcor_v_others)
            perc25_dcor = np.percentile(dcor_v_others, 25)
            perc75_dcor = np.percentile(dcor_v_others, 75)
        else:
            max_dcor = min_dcor = mean_dcor = std_dcor = perc25_dcor = perc75_dcor = 0.0

        # Append the computed features to the list
        df_list.append({
            "variable": variable,
            "dcor(v,X)": dcor_v_X,
            "dcor(v,Y)": dcor_v_Y,
            "dcor(v,X)^2": dcor_v_X_sq,
            "dcor(v,Y)^2": dcor_v_Y_sq,
            "max(dcor(v, others))": max_dcor,
            "min(dcor(v, others))": min_dcor,
            "mean(dcor(v, others))": mean_dcor,
            "std(dcor(v, others))": std_dcor,
            "25%(dcor(v, others))": perc25_dcor,
            "75%(dcor(v, others))": perc75_dcor,
            "dcor(X,Y)": dcor_X_Y,
            "dcor(X,Y)^2": dcor_X_Y_sq
        })

    # Convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(df_list)

    # Add dataset name if available
    df["dataset"] = getattr(dataset, "name", "Unknown Dataset")

    # Reorder columns to have 'dataset' first
    cols = ["dataset"] + [col for col in df.columns if col != "dataset"]
    df = df[cols]

    return df

### Information-Theoretic Features

- Mutual Information
- Conditional Mutual Information
- Copula Entropy - pycopent def copula_entropy() with copula_entropy(v, X and Y)

In [15]:
def mutual_information_and_conditional_MI(dataset):
    """
    Computes mutual-information-based features for each variable, including mutual
    information with X and Y, summary statistics of MI with other variables, and
    estimates of conditional mutual information.
    
    Conditional Mutual Information Calculations:
    - CMI(v, X | Y)
    - CMI(v, Y | X)
    - CMI(X, Y | v)
    """
    variables = dataset.columns.drop(["X", "Y"])
    df_list = []

    # Initialize the imputer to fill NaN values with the mean of each column
    imputer = SimpleImputer(strategy='mean')
    imputed_data = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)


    scaler = StandardScaler()
    scaled_data = pd.DataFrame(scaler.fit_transform(imputed_data), columns=dataset.columns)

    # Compute MI of all variables with X and Y at once
    mi_with_X = mutual_info_regression(scaled_data[variables], scaled_data["X"], discrete_features=False)
    mi_with_Y = mutual_info_regression(scaled_data[variables], scaled_data["Y"], discrete_features=False)

    # Compute MI between each variable and all other variables
    mi_matrix = pd.DataFrame(index=variables, columns=variables)
    for var in variables:
        mi_matrix[var] = mutual_info_regression(scaled_data[variables], scaled_data[var], discrete_features=False)

    # Compute MI between X and Y
    mi_XY = mutual_info_regression(scaled_data[["X"]], scaled_data["Y"], discrete_features=False)[0]

    for idx, variable in enumerate(variables):
        # Get MI(v, others)
        mi_v_others = mi_matrix[variable].drop(variable)

        # Append results
        result = {
            "variable": variable,
            "MI(v,X)": mi_with_X[idx],
            "MI(v,Y)": mi_with_Y[idx],
            "max(MI(v, others))": mi_v_others.max(),
            "min(MI(v, others))": mi_v_others.min(),
            "mean(MI(v, others))": mi_v_others.mean(),
            "std(MI(v, others))": mi_v_others.std(),
        }

        # Conditional Mutual Information Calculations
        # CMI(v, X | Y) = MI(v, X, Y) - MI(v, Y)
        try:
            mi_v_X_Y = mutual_info_regression(
                scaled_data[[variable, "Y"]], scaled_data["X"], discrete_features=False
            )[0]
            cmi_v_x_given_y = mi_v_X_Y - mi_with_Y[idx]
        except:
            cmi_v_x_given_y = np.nan

        # CMI(v, Y | X) = MI(v, Y, X) - MI(v, X)
        try:
            mi_v_Y_X = mutual_info_regression(
                scaled_data[[variable, "X"]], scaled_data["Y"], discrete_features=False
            )[0]
            cmi_v_y_given_x = mi_v_Y_X - mi_with_X[idx]
        except:
            cmi_v_y_given_x = np.nan

        # CMI(X, Y | v) = MI(X, Y, v) - MI(X, v)
        try:
            mi_X_Y_v = mutual_info_regression(
                scaled_data[[ "X", variable]], scaled_data["Y"], discrete_features=False
            )[0]
            mi_X_v = mutual_info_regression(
                scaled_data[[variable]], scaled_data["X"], discrete_features=False
            )[0]
            cmi_x_y_given_v = mi_X_Y_v - mi_X_v
        except:
            cmi_x_y_given_v = np.nan

        # Add Conditional MI to the result
        result["conditional_MI(v,X|Y)"] = cmi_v_x_given_y
        result["conditional_MI(v,Y|X)"] = cmi_v_y_given_x
        result["conditional_MI(X,Y|v)"] = cmi_x_y_given_v

        df_list.append(result)

    df = pd.DataFrame(df_list)
    df["dataset"] = getattr(dataset, "name", "Unknown Dataset")
    df["MI(X,Y)"] = mi_XY

    # Reorder columns: dataset first, then MI metrics
    mi_columns = [col for col in df.columns if col not in ["dataset"]]
    df = df[["dataset"] + mi_columns]

    return df

In [16]:
#copula_entropy - copula_entropy(v,X)、copula_entropy(v,Y)

def emp_u_of_optimized(x):
    sorted_indices = np.argsort(x)
    ranks = np.empty_like(sorted_indices)
    ranks[sorted_indices] = np.arange(1, len(x) + 1)
    return ranks / len(x)

def kraskov_entropy(data, k=3, epsilon=1e-10):
    """
    Estimate the entropy using the Kraskov method.
    Parameters:
    - data: Array of shape (n_samples, n_features).
    - k: Number of nearest neighbors.
    - epsilon: Small value added to avoid log(0).
    """
    n, d = data.shape
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(data)
    distances, _ = nbrs.kneighbors(data)
    
    # Distance to the k-th nearest neighbor
    epsilons = distances[:, k]
    
    # Avoid log(0) by adding epsilon and clipping
    epsilons = np.clip(epsilons, epsilon, None)
    
    avg_log_eps = np.mean(np.log(epsilons))
    const = np.log(n - 1) + np.log(2 * np.pi) * (d / 2) - np.log(k) - 1
    return const + d * avg_log_eps

def emp_Hc_of(x, y, k=3):
    ux, uy = emp_u_of_optimized(x), emp_u_of_optimized(y)
    u = np.column_stack((ux, uy))
    return kraskov_entropy(u, k=k)

x = np.random.rand(100)
y = np.random.rand(100)

# Compute empirical copula entropy
Hc = emp_Hc_of(x, y, k=3)
print(f"Empirical Copula Entropy: {Hc}")

def compute_copula_features(dataset):
    """
    Given a dataset, this function computes the copula entropy between
    each variable v and X, Y. The resulting copula entropy values are 
    used to create features to describe/embed v, as well as the copula 
    entropy between X and Y.
    """
    variables = dataset.columns.drop(["X", "Y"])

    df = []
    for variable in variables:
        copula_vX = emp_Hc_of(dataset[variable].values, dataset["X"].values)
        copula_vY = emp_Hc_of(dataset[variable].values, dataset["Y"].values)

        df.append({
            "variable": variable,
            "copula_entropy(v,X)": copula_vX,
            "copula_entropy(v,Y)": copula_vY,
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    copula_XY = emp_Hc_of(dataset["X"].values, dataset["Y"].values)
    df["copula_entropy(X,Y)"] = copula_XY

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

Empirical Copula Entropy: -0.2597106224714887


In [17]:
def kraskov_entropy(data, k=3, epsilon=1e-10):
    """
    Estimate the entropy using the Kraskov method.
    Parameters:
    - data: Array of shape (n_samples, n_features).
    - k: Number of nearest neighbors.
    - epsilon: Small value added to avoid log(0).
    """
    n, d = data.shape
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(data)
    distances, _ = nbrs.kneighbors(data)
    
    # Distance to the k-th nearest neighbor
    epsilons = distances[:, k]
    
    # Avoid log(0) by adding epsilon and clipping
    epsilons = np.clip(epsilons, epsilon, None)
    
    avg_log_eps = np.mean(np.log(epsilons))
    const = np.log(n - 1) + np.log(2 * np.pi) * (d / 2) - np.log(k) - 1
    return const + d * avg_log_eps

def joint_entropy(x, y, k=3, epsilon=1e-10):
    data = np.column_stack((x, y))
    return kraskov_entropy(data, k=k, epsilon=epsilon)

def conditional_entropy(x, y, k=3, epsilon=1e-10):
    data = np.column_stack((x, y))
    joint_ent = kraskov_entropy(data, k=k, epsilon=epsilon)
    marginal_y = kraskov_entropy(y.reshape(-1, 1), k=k, epsilon=epsilon)
    return joint_ent - marginal_y

def compute_information_metrics(dataset, k=3, epsilon=1e-10):
    variables = dataset.columns.drop(["X", "Y"])
    
    df = []
    for variable in variables:
        var = dataset[variable].values
        x = dataset["X"].values
        y = dataset["Y"].values
        
        # Joint entropies
        jh_vx = joint_entropy(var, x, k=k, epsilon=epsilon)
        jh_vy = joint_entropy(var, y, k=k, epsilon=epsilon)
        
        # Conditional entropies
        ch_vx = conditional_entropy(var, x, k=k, epsilon=epsilon)
        ch_vy = conditional_entropy(var, y, k=k, epsilon=epsilon)
        
        df.append({
            "variable": variable,
            "JH(v,X)": jh_vx,
            "JH(v,Y)": jh_vy,
            "CH(v|X)": ch_vx,
            "CH(v|Y)": ch_vy,
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    # Reorder columns
    df = df[["dataset"] + [col for col in df.columns if col != "dataset"]]
    
    return df

### Discretization-Based Features

> For Features 1-3, the original variables after discretization are input; for Features 4-7, the normalized discrete variables are input.
> 

> Reference: https://github.com/diogo149/CauseEffectPairsChallenge
> 
1. **Information Metrics**
    - JH(v,x): The joint entropy $H(v, x)$ of variable *v* and *X*.
    - JH(v,y): The joint entropy $H(v, y)$ of variable *v* and *Y*.
    - CH(v|x): The conditional entropy $H(v|x)$ of variable *v* given *X*.
    - CH(v|y): The conditional entropy $H(v|y)$ of variable *v* given *Y*.
2. **Mutual Information Metrics**
    - I(v,x): The mutual information $I(v; x)$ between variable *v* and *X*.
    - I(v,y): The mutual information $I(v; y)$ between variable *v* and *Y*.
    - AMI(v,x): The adjusted mutual information $AMI(v; x)$ between variable *v* and *X*.
    - AMI(v,y): The adjusted mutual information $AMI(v; y)$ between variable *v* and *Y*.
3. **Error Probability Metrics**
    - NPE(v,x): The normalized prediction error between variable *v* and *X*.
    - NPE(v,y): The normalized prediction error between variable *v* and *Y*.
4. **Information-Geometric Causal Inference (IGCI) Metrics**
    - IGCI(v,x): The IGCI metric based on *v* and *X*.
    - IGCI(v,y): The IGCI metric based on *v* and *Y*.
    - IGCI(x,v): The IGCI metric based on *X* and *v*.
    - IGCI(y,v): The IGCI metric based on *Y* and *v*.
5. **Moment Features**
    - MM(v² * x): The normalized joint moment $I(v^2 * x)$.
    - MM(v² * y): The normalized joint moment $I(v^2 * y)$.
    - MM(v² * x²): The normalized joint moment $I(v^2 * x^2)$.
    - MM(v² * y²): The normalized joint moment $I(v^2 * y^2)$.
    - MM(v³ * x): The normalized joint moment $I(v^3 * x)$.
    - MM(v³ * y): The normalized joint moment $I(v^3 * y)$.
6. **Fit Metrics**
    - fit(v,x): The fit metric based on *v* and *X*, calculated via polynomial fitting.
    - fit(v,y): The fit metric based on *v* and *Y*, calculated via polynomial fitting.
    - fit_error(v,x): The fit error based on *v* and *X*, measured by the standard deviation of residuals after polynomial fitting.
    - fit_error(v,y): The fit error based on *v* and *Y*, measured by the standard deviation of residuals after polynomial fitting.
    - fit_noise_entropy(v,x): The noise entropy after fitting, measuring the information content of noise for *v* and *X*.
    - fit_noise_entropy(v,y): The noise entropy after fitting, measuring the information content of noise for *v* and *Y*.
    - fit_noise_skewness(v,x): The standard deviation of noise skewness, measuring asymmetry in the noise distribution for *v* and *X*.
    - fit_noise_skewness(v,y): The standard deviation of noise skewness, measuring asymmetry in the noise distribution for *v* and *Y*.
    - fit_noise_kurtosis(v,x): The standard deviation of noise kurtosis, measuring the peakedness of the noise distribution for *v* and *X*.
    - fit_noise_kurtosis(v,y): The standard deviation of noise kurtosis, measuring the peakedness of the noise distribution for *v* and *Y*.
7. **Conditional Distribution Similarity**
    - cond_dist_sim(v,x): The conditional distribution similarity of variable *v* relative to *X*.
    - cond_dist_sim(v,y): The conditional distribution similarity of variable *v* relative to *Y*.

In [20]:
def compute_information_metrics(dataset, bins=10):
    """
    Computes information metrics for each variable in the dataset with respect to 'X' and 'Y':
    1. JH(v,X): The joint entropy H(v, X) of variable v and X.
    2. JH(v,Y): The joint entropy H(v, Y) of variable v and Y.
    3. CH(v|X): The conditional entropy H(v|X) of variable v given X.
    4. CH(v|Y): The conditional entropy H(v|Y) of variable v given Y.
    """
    variables = dataset.columns.drop(['X', 'Y'], errors='ignore')
    df_list = []

    # Handle missing values by imputing with the mean
    imputer = SimpleImputer(strategy='mean')
    imputed_data = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

    # Discretize 'X' and 'Y' into bins
    try:
        x_disc = pd.cut(imputed_data['X'], bins=bins, labels=False)
        y_disc = pd.cut(imputed_data['Y'], bins=bins, labels=False)
    except KeyError:
        x_disc = y_disc = pd.Series([np.nan]*len(imputed_data))
        print("Warning: 'X' or 'Y' not found in the dataset.")

    for variable in variables:
        try:
            v_disc = pd.cut(imputed_data[variable], bins=bins, labels=False)

            # Remove any NaN values resulting from discretization
            df = pd.DataFrame({'v': v_disc, 'X': x_disc, 'Y': y_disc}).dropna()

            # Joint entropy H(v,X)
            joint_vx_counts = df.groupby(['v', 'X']).size().values
            jh_vx = entropy(joint_vx_counts, base=2)

            # Joint entropy H(v,Y)
            joint_vy_counts = df.groupby(['v', 'Y']).size().values
            jh_vy = entropy(joint_vy_counts, base=2)

            # Entropy H(X)
            x_counts = df['X'].value_counts().values
            h_x = entropy(x_counts, base=2)

            # Entropy H(Y)
            y_counts = df['Y'].value_counts().values
            h_y = entropy(y_counts, base=2)

            # Conditional entropy H(v|X) = H(v,X) - H(X)
            ch_v_given_x = jh_vx - h_x

            # Conditional entropy H(v|Y) = H(v,Y) - H(Y)
            ch_v_given_y = jh_vy - h_y

            result = {
                'variable': variable,
                'JH(v,X)': jh_vx,
                'JH(v,Y)': jh_vy,
                'CH(v|X)': ch_v_given_x,
                'CH(v|Y)': ch_v_given_y,
            }
        except Exception as e:
            # Handle exceptions and assign 0 to results
            result = {
                'variable': variable,
                'JH(v,X)': 0,
                'JH(v,Y)': 0,
                'CH(v|X)': 0,
                'CH(v|Y)': 0,
            }
            print(f"Error processing variable '{variable}': {e}")

        df_list.append(result)

    # Convert results to DataFrame
    df_results = pd.DataFrame(df_list)

    # Add dataset name if available
    df_results['dataset'] = getattr(dataset, 'name', 'Unknown Dataset')

    # Reorder columns
    columns = ['dataset', 'variable', 'JH(v,X)', 'JH(v,Y)', 'CH(v|X)', 'CH(v|Y)']
    df_results = df_results[columns]

    return df_results

In [21]:
def energy_distance_features(dataset):
    """
    Given a dataset, we compute the energy distance-based features for each
    variable, which are the energy distance between that variable with X and Y,
    as well as summary statistics (max, min, mean, std) of all pairs
    of energy distances.
    """

    variables = dataset.columns.drop(["X", "Y"])
    df = []
    
    for variable in variables:
        #energy_dists = dataset.apply(lambda col: energy_distance(dataset[variable], col))
        #energy_dists = dataset.apply(lambda col: energy_distance(dataset[variable].values[0:10], col.values[0:10]))
        energy_dists_dict = {}
        target_values = dataset[variable].values[0:-1]
        for col_name in dataset.columns:
            energy_dists_dict[col_name] = energy_distance(target_values, dataset[col_name].values[0:-1])
            
        energy_dists = pd.Series(energy_dists_dict)

        #energy_dists = energy_dists.drop([variable, "X", "Y"])

        energy_dists = energy_dists.fillna(0)

        df.append({
            "variable": variable,
            "energy_dist(v,X)": energy_distance(dataset[variable].values[0:-1], dataset["X"].values[0:-1]),
            "energy_dist(v,Y)": energy_distance(dataset[variable].values[0:-1], dataset["Y"].values[0:-1]),
            "max(energy_dist(v, others))": energy_dists.max(),
            "min(energy_dist(v, others))": energy_dists.min(),
            "mean(energy_dist(v, others))": energy_dists.mean(),
            "std(energy_dist(v, others))": energy_dists.std(),
            "std25(energy_dist(v, others))": np.percentile(energy_dists.values, 25),
            "std75(energy_dist(v, others))": np.percentile(energy_dists.values, 75),
            "median(energy_dist(v, others))": np.median(energy_dists.values)
            #"energy_dist(X,Y)": energy_dist_x_y
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    df["energy_dist(X,Y)"] = energy_distance(dataset["X"].values[0:-1], dataset["Y"].values[0:-1])

    df.fillna(0, inplace=True)

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]
    return df


In [13]:
if False:
    i=0
    for name, dataset in tqdm(X_train.items()):
        adataset = X_train[name]
        adataset.name = name
        if name == '00067':
            df_dataset = partial_correlation(adataset) #spearman_correlation(dataset) #pearson_correlation_with_rolling_multi_labeled(dataset)
            print("very good!")


In [36]:
def ttest(dataset, pvalue_threshold=0.05):
    """
    Given a dataset, this function computes the t-test between the
    values each variable v and X, Y. The t value and the result of the
    t-test with a given pvalue_threshold, are used to create features
    to describe/embed v, as well as the t-test result between the
    values of X and Y.
    """

    variables = dataset.columns.drop(["X", "Y"])

    df = []
    for variable in variables:
        ttest_vX = ttest_rel(dataset[variable], dataset["X"])
        ttest_vY = ttest_rel(dataset[variable], dataset["Y"])

        df.append({
            "variable": variable,
            "ttest(v,X)": ttest_vX.statistic,
            f"pvalue(ttest(v,X))<={pvalue_threshold}": (ttest_vX.pvalue <= pvalue_threshold).astype(float),
            "ttest(v,Y)": ttest_vY.statistic,
            f"pvalue(ttest(v,Y))<={pvalue_threshold}": (ttest_vY.pvalue <= pvalue_threshold).astype(float),
        })

    df = pd.DataFrame(df)
    df["dataset"] = dataset.name

    ttest_XY = ttest_rel(dataset["X"], dataset["Y"])
    df["ttest(X,Y)"] = ttest_XY.statistic
    df[f"pvalue(ttest(X,Y))<={pvalue_threshold}"] = (ttest_XY.pvalue <= pvalue_threshold).astype(float)

    # some the ttest returns NaN when the variance is 0, so we fill with 0:
    df.fillna(0, inplace=True)

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

In [22]:
if False:
    data = {
        "X": np.random.rand(100),
        "Y": np.random.rand(100),
        "v1": np.random.rand(100),
        "v2": np.random.rand(100),
    }
    dataset = pd.DataFrame(data)
    dataset.name = "example_dataset"

    # Compute information metrics
    info_metrics = compute_information_metrics(dataset)
    print(info_metrics)

In [23]:
def label(adjacency_matrix):
    """
    Given a graph as adjacency_matrix, create the class labels of each variable.
    """

    adjacency_graph, adjacency_label = create_graph_label()
    labels = get_labels(adjacency_matrix, adjacency_label)
    variables = adjacency_matrix.columns.drop(["X", "Y"])

    df = pd.DataFrame({
        "variable": variables,
        "label": [labels[variable] for variable in variables],
    })
    df["dataset"] = adjacency_matrix.name

    # Reorder columns:
    df = df[["dataset"] + [colname for colname in df.columns if colname != "dataset"]]

    return df

In [24]:
def create_some_columns(names_datasets, function):
    """
    Apply an embedding function to a list of datasets.
    """

    df = []
    for name, dataset in tqdm(names_datasets.items()):
        dataset = names_datasets[name]
        dataset.name = name

        try:
            df_dataset = function(dataset)
        except ValueError as e:
            print(name, e)
            raise NotImplementedError

        df_dataset["dataset"] = name
        df.append(df_dataset)

    df = pd.concat(df, axis="index").reset_index(drop=True)
    return df

In [25]:
def create_some_columns_parallel(names_datasets, function, n_jobs=-1):
    """
    Apply an embedding function to a list of datasets.

    Parallel version.
    """

    def f(name, dataset, function):
        dataset.name = name
        df_dataset = function(dataset)
        df_dataset["dataset"] = name
        return df_dataset

    df = joblib.Parallel(n_jobs=n_jobs)(
        joblib.delayed(f)(name, dataset, function)
        for name, dataset in tqdm(names_datasets.items())
    )

    df = pd.concat(df, axis="index").reset_index(drop=True)
    return df

In [26]:
def create_all_columns(functions_names_datasets, n_jobs=-1):
    """
    given a dictionary of {function1:names, function2:names,...} apply
    the desired functions to the list of datasets and merge all of them
    in a single X_y_group dataframe.
    """

    columns = []
    for function, names_datasets in functions_names_datasets.items():
        print(f"set: {function.__name__}")

        if n_jobs != 1:
            feature_set = create_some_columns_parallel(names_datasets, function, n_jobs=n_jobs)
        else:
            feature_set = create_some_columns(names_datasets, function)

        columns.append(feature_set)

    # Merge all feature sets into a single dataframe:
    columns = functools.reduce(
        lambda left, right: pd.merge(left, right, on=["dataset", "variable"]),
        columns,
    )

    return columns

We now compute `(X_train, y_train)` from all training datasets and graphs, and store it in a single dataframe `X_y_group_train`, which also contains the name of the dataset to which each variable belongs.

In Google Colab, the cell below takes about half an hour to compute. We keep `n_jobs=1` to avoid excessive memory usage.

In [ ]:
if False:
    names_datasets_train = X_train
    names_graphs_train = y_train

    print(f"Creating X_y_group_train from {len(names_datasets_train)} datasets and graphs")
    X_y_group_train = create_all_columns(
        {
            distance_correlation_features: names_datasets_train,
            pearson_correlation: names_datasets_train,
            spearman_correlation: names_datasets_train,
            kendall_correlation: names_datasets_train,
            pearson_correlation_with_rolling_multi_labeled: names_datasets_train,
            partial_correlation: names_datasets_train,
            partial_correlation_rank:  names_datasets_train,
            mutual_information_and_conditional_MI: names_datasets_train,
            compute_copula_features: names_datasets_train,
            energy_distance_features: names_datasets_train,
            ttest: names_datasets_train,
            label: names_graphs_train,
            compute_information_metrics: names_datasets_train,
        },
        n_jobs=-1,
    )

Creating X_y_group_train from 23500 datasets and graphs
set: distance_correlation_features


100%|██████████| 23500/23500 [09:12<00:00, 42.52it/s]


In [ ]:
create_feature_file = False
if create_feature_file:
    with open('X_y_group_train.pkl', 'wb') as f:
        pickle.dump(X_y_group_train, f)

    X_y_group_train

In [72]:
print("Adding numeric labels y")
le = LabelEncoder()
X_y_group_train["y"] = le.fit_transform(X_y_group_train["label"])
# reordering columns:
X_y_group_train = X_y_group_train[["dataset", "variable"] + X_y_group_train.columns.drop(["dataset", "variable", "label", "y"]).tolist() + ["label", "y"]]
#display(X_y_group_train)

print("Extracting X_train, y_train, and group")
X_train_feat = X_y_group_train.drop(["variable", "dataset", "label", "y"], axis="columns")
y_train_feat = X_y_group_train["y"]
group_train = X_y_group_train["dataset"]
print(X_train_feat.shape)

Adding numeric labels y
Extracting X_train, y_train, and group
(142910, 85)


In [ ]:
#GAE
# A gradient-based algorithm using graph autoencoder to model non-linear causal relationships

#weighted_random_dag = DAG.erdos_renyi(n_nodes=10, n_edges=20, weight_range=(0.5, 2.0), seed=1)
#dataset = IIDSimulation(W=weighted_random_dag, n=2000, method='linear', sem_type='gauss')
#true_dag, X = dataset.B, dataset.X

if False:
    A = names_graphs_train['00016']
    print(X_train)
    X = X_train['00016']
    print("y_true:", A)
    print("X:", X.head())

    ga = GAE(input_dim=10)
    ga.learn(X)

    # plot est_dag and true_dag
    GraphDAG(ga.causal_matrix, true_dag)

    # calculate accuracy
    met = MetricsDAG(ga.causal_matrix, true_dag)
    print(met.metrics)

In [29]:
if False:
    #shape 16
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=11,
        n_jobs=-1,
        class_weight="balanced"
    )
    display(model)

    print("X_train shape:", X_train_feat.shape)
    print("X_train data types:", X_train_feat.dtypes)
    print("y_train shape:", y_train_feat.shape)
    print("y_train data type:", y_train_feat.dtype)
    print("group_train shape:", group_train.shape)
    print("group_train data type:", group_train.dtype)

    cv = GroupKFold(n_splits=4)
    results = cross_val_score(
        model,
        X_train_feat,
        y_train_feat,
        groups=group_train,
        cv=cv,
        verbose=True,
        scoring="balanced_accuracy"
    )

    print(f"multiclass balanced accuracy: mean={results.mean()}")

In [27]:
#print(X_train_feat.shape)
#X_train_feat.columns

In [28]:
if False:

    # Clean feature names
    X_train_feat.columns = [col.replace(' ', '_').replace('"', '').replace("'", '') for col in X_train_feat.columns]
    X_train_feat = X_train_feat.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

    # Check for constant features
    constant_features = [col for col in X_train_feat.columns if X_train_feat[col].nunique() == 1]
    if constant_features:
        print(f"Constant features: {constant_features}")
        X_train_feat = X_train_feat.drop(columns=constant_features)

    # Check for missing values
    missing_values = X_train_feat.isnull().sum()
    if missing_values.any():
        print(f"Missing values: {missing_values[missing_values > 0]}")
        X_train_feat = X_train_feat.fillna(0)  # Fill missing values with 0 or use an appropriate strategy

    # Define the model
    model = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=11,
        n_jobs=-1,
        class_weight="balanced",
        verbose=0,
        min_child_samples=20,  # Minimum number of data points in a leaf
        min_split_gain=0.01,   # Minimum gain to make a split
        learning_rate=0.1      # Learning rate
    )
    display(model)

    # Print the shapes of the training data
    print("X_train shape:", X_train_feat.shape)
    print("y_train shape:", y_train_feat.shape)
    print("group_train shape:", group_train.shape)

    # Define the cross-validation strategy
    cv = GroupKFold(n_splits=4)

    # Perform cross-validation
    results = cross_val_score(
        model,
        X_train_feat,
        y_train_feat,
        groups=group_train,
        cv=cv,
        verbose=True,
        scoring="balanced_accuracy"
    )

    # Print the mean balanced accuracy
    print(f"Multiclass balanced accuracy: mean={results.mean()}")

In [30]:
if False:
    x16 = X_train['00016']
    y16 = y_train['00016']

    count = 0
    for name, dataset in X_train.items():
        # Check if the dataset has exactly three columns named 'X', 'Y', and '0'
        if set(dataset.columns) == {'X', 'Y', '0'}:
            print(f"First dataset with columns 'X', 'Y', and '0' is: {name}")
            x16 = dataset
            if count == 2:
                break
            count += 1

    sorted_x16 = x16.sort_values(by='0').reset_index(drop=True)

    plt.figure(figsize=(10, 6))
    plt.plot(sorted_x16['0'], sorted_x16['0'], marker='o', label='Y')
    plt.plot(sorted_x16['0'], sorted_x16['Y'], marker='o', label='Y')
    plt.plot(sorted_x16['0'], sorted_x16['X'], marker='o', label='X')
    plt.xlabel('Sorted Variable 0')
    plt.ylabel('Y')
    plt.title('Sorted Variable 0 vs Y')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Print the sorted DataFrame
    print(sorted_x16)

    print(x16.corr())

In [37]:
names_datasets_train = X_train
names_graphs_train = y_train
# List of feature functions and their corresponding datasets
feature_functions = {
    distance_correlation_features: names_datasets_train,
    pearson_correlation: names_datasets_train,
    spearman_correlation: names_datasets_train,
    kendall_correlation: names_datasets_train,
    pearson_correlation_with_rolling_multi_labeled: names_datasets_train,
    partial_correlation: names_datasets_train,
    partial_correlation_rank: names_datasets_train,
    mutual_information_and_conditional_MI: names_datasets_train,
    compute_copula_features: names_datasets_train,
    energy_distance_features: names_datasets_train,
    ttest: names_datasets_train,
    compute_information_metrics: names_datasets_train,
}

# Create the full dataset with all features
print(f"Creating X_y_group_train from {len(names_datasets_train)} datasets and graphs")
X_y_group_train = create_all_columns(
    {**feature_functions, label: names_graphs_train}, n_jobs=-1
)

# Ensure feature names are unique and sanitize them
X_y_group_train.columns = pd.Index([f"{col}_{i}" if X_y_group_train.columns.duplicated()[i] else col for i, col in enumerate(X_y_group_train.columns)])
X_y_group_train.columns = X_y_group_train.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)

# Adding numeric labels y
print("Adding numeric labels y")
le = LabelEncoder()
X_y_group_train["y"] = le.fit_transform(X_y_group_train["label"])

# Reordering columns
X_y_group_train = X_y_group_train[
    ["dataset", "variable"] +
    X_y_group_train.columns.drop(["dataset", "variable", "label", "y"]).tolist() +
    ["label", "y"]
]

# Extracting features and labels
print("Extracting X_train, y_train, and group")
X_train_feat = X_y_group_train.drop(["variable", "dataset", "label", "y"], axis="columns")
y_train_feat = X_y_group_train["y"]
group_train = X_y_group_train["dataset"]

print("X_train shape:", X_train_feat.shape)
print("X_train data types:", X_train_feat.dtypes)
print("y_train shape:", y_train_feat.shape)
print("y_train data type:", y_train_feat.dtype)
print("group_train shape:", group_train.shape)
print("group_train data type:", group_train.dtype)

# Check for randomness in group identifiers using the runs test
print("\nChecking for randomness in group identifiers using the runs test")
group_ids = group_train.values
differences = np.diff(group_ids)
print("Differences between consecutive group_id values:")
print(differences)

# Convert differences to binary sequence for runs test
binary_sequence = np.sign(differences)
binary_sequence[binary_sequence == 0] = 1  # Replace zeros with ones to avoid issues

# Perform the runs test
z_stat, p_value = runstest_1samp(binary_sequence)
print(f"Runs test z-statistic: {z_stat}")
print(f"Runs test p-value: {p_value}")

# Interpretation
if p_value < 0.05:
    print("The sequence of group identifiers is not random (p < 0.05).")
else:
    print("The sequence of group identifiers is random (p >= 0.05).")

# Initialize the model
model = lgb.LGBMClassifier()

print("\nModel:")
print(model)

# Perform cross-validation with grouping
cv = GroupKFold(n_splits=4)
results = []

for train_index, test_index in cv.split(X_train_feat, y_train_feat, groups=group_train):
    X_train_cv, X_test_cv = X_train_feat.iloc[train_index], X_train_feat.iloc[test_index]
    y_train_cv, y_test_cv = y_train_feat.iloc[train_index], y_train_feat.iloc[test_index]

    model.fit(X_train_cv, y_train_cv)
    pred = model.predict_proba(X_test_cv)[:, 1]  # Adjust indexing for multiclass
    score = roc_auc_score(y_test_cv, pred, multi_class='ovr')
    results.append(score)
    print(f"Fold ROC AUC Score: {score}")

mean_score = np.mean(results)
print(f"\nMulticlass ROC AUC Score: mean={mean_score}")

# Dictionary to store performance results
feature_importance = {}

# Perform leave-one-out feature importance
for function, datasets in feature_functions.items():
    print(f"\nEvaluating feature importance without: {function.__name__}")

    # Create dataset excluding features from the current function
    functions_to_include = {
        func: data for func, data in feature_functions.items() if func != function
    }

    X_y_group_train_reduced = create_all_columns(
        {**functions_to_include, label: names_graphs_train}, n_jobs=-1
    )

    # Adding numeric labels y
    X_y_group_train_reduced["y"] = le.transform(X_y_group_train_reduced["label"])

    # Reordering columns
    X_y_group_train_reduced = X_y_group_train_reduced[
        ["dataset", "variable"] +
        X_y_group_train_reduced.columns.drop(["dataset", "variable", "label", "y"]).tolist() +
        ["label", "y"]
    ]

    # Extracting features and labels
    X_reduced_feat = X_y_group_train_reduced.drop(["variable", "dataset", "label", "y"], axis="columns")
    y_reduced_feat = X_y_group_train_reduced["y"]
    group_reduced = X_y_group_train_reduced["dataset"]

    # Cross-validation
    results_reduced = []

    for train_index, test_index in cv.split(X_reduced_feat, y_reduced_feat, groups=group_reduced):
        X_train_cv, X_test_cv = X_reduced_feat.iloc[train_index], X_reduced_feat.iloc[test_index]
        y_train_cv, y_test_cv = y_reduced_feat.iloc[train_index], y_reduced_feat.iloc[test_index]

        model.fit(X_train_cv, y_train_cv)
        pred = model.predict_proba(X_test_cv)[:, 1]  # Adjust indexing for multiclass
        score = roc_auc_score(y_test_cv, pred, multi_class='ovr')
        results_reduced.append(score)

    mean_score_reduced = np.mean(results_reduced)
    print(f"Performance without {function.__name__}: {mean_score_reduced}")

    # Compute the performance drop
    performance_drop = mean_score - mean_score_reduced
    feature_importance[function.__name__] = performance_drop

# Display feature importance
importance_df = pd.DataFrame.from_dict(
    feature_importance, orient='index', columns=['Performance Drop']
).sort_values(by='Performance Drop', ascending=False)

print("\nFeature Importance based on Performance Drop:")
print(importance_df)

Creating X_y_group_train from 23500 datasets and graphs
set: distance_correlation_features


100%|██████████| 23500/23500 [09:40<00:00, 40.47it/s]


set: pearson_correlation


100%|██████████| 23500/23500 [01:04<00:00, 365.46it/s]


set: spearman_correlation


100%|██████████| 23500/23500 [01:24<00:00, 276.71it/s]


... (7 outputs left out)

  0%|          | 32/23500 [00:00<01:37, 241.69it/s]/home/endrem/prosjekt/causalDiscovery/causalEnv/lib/python3.12/site-packages/pingouin/correlation.py:59: RuntimeWarning: divide by zero encountered in scalar divide
  tval = r * np.sqrt(dof / (1 - r**2))
/home/endrem/prosjekt/causalDiscovery/causalEnv/lib/python3.12/site-packages/pingouin/effsize.py:152: RuntimeWarning: divide by zero encountered in arctanh
  z = np.arctanh(stat)  # R-to-z transform
/home/endrem/prosjekt/causalDiscovery/causalEnv/lib/python3.12/site-packages/pingouin/correlation.py:59: RuntimeWarning: divide by zero encountered in scalar divide
  tval = r * np.sqrt(dof / (1 - r**2))
/home/endrem/prosjekt/causalDiscovery/causalEnv/lib/python3.12/site-packages/pingouin/effsize.py:152: RuntimeWarning: divide by zero encountered in arctanh
  z = np.arctanh(stat)  # R-to-z transform
/home/endrem/prosjekt/causalDiscovery/causalEnv/lib/python3.12/site-packages/pingouin/correlation.py:59: RuntimeWarning: divide by zero encount

set: mutual_information_and_conditional_MI


 63%|██████▎   | 14880/23500 [42:52<26:11,  5.48it/s] 

KeyboardInterrupt: 

In [38]:
from random import seed, sample
seed(0,1)

x=list(range(1000))
y=list(reversed(x))
z=sample(range(2000), 1000)

pg.partial_corr(data=pd.DataFrame(dict(x=x, y=y, z=z)), x='x', y='y', covar='z', method='spearman')


,n,r,CI95%,p-val
spearman,1000,1.0,"[1.0, 1.0]",0.0
